# Clase 07 · De los documentos al índice
**Inteligencia Artificial Generativa y Sistemas Inteligentes · EAM 2026-II**

**Proyecto: Buscador de conocimiento** (Joan Sebastián Sosa Bedoya · Alejandra Mejía Patiño). Objetivo: recuperar contenido por significado y entregar resultados pertinentes con referencia al documento de origen.

En esta práctica prepararemos dos documentos del proyecto, compararemos dos formas de fragmentarlos y guardaremos un índice para volver a consultarlo. Trabajaremos en Colab, con el mismo modelo de embeddings de la clase anterior. La primera ejecución necesita Internet para instalar la biblioteca y descargar el modelo; podemos usar CPU.

Ejecuten las celdas en orden. Las políticas del proyecto son **ficticias**, igual que el repositorio de documentos de la Clase 4. La consulta se escribe en el punto **5**.

## 1. Preparación
Instalamos la biblioteca y cargamos el modelo. Comprobamos su límite de entrada para evitar que el texto se recorte al generar los embeddings.

In [ ]:
%pip -q install sentence-transformers==6.1.0

In [ ]:
from pathlib import Path
import re, json, hashlib, zipfile
import numpy as np
from sentence_transformers import SentenceTransformer
from huggingface_hub import model_info

MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
REVISION = model_info(MODELO).sha
modelo = SentenceTransformer(MODELO, revision=REVISION)
tokenizador = modelo.tokenizer
MAX_TOKENS = 80       # Presupuesto del fragmento completo, incluido el encabezado.
SOLAPAMIENTO = 12    # Tokens repetidos entre ventanas de un mismo bloque.

def contar_tokens(texto):
    return len(tokenizador.encode(texto, add_special_tokens=True, truncation=False))

print("Límite del modelo:", modelo.max_seq_length, "tokens")
print("Presupuesto de la práctica:", MAX_TOKENS, "tokens")

## 2. Los documentos del corpus
Usamos dos documentos del repositorio del proyecto (Clase 4): `Politica_Recursos_Humanos` (**DOC-1001**, vigente) y `Politica_Permisos` (**DOC-1002**, en revisión). Conservan los fragmentos que ya usaba el modo mock del Corte 1. `Manual_Induccion` (DOC-1003) queda **excluido** porque su estado es desactualizado.

Esta celda crea dos archivos Markdown en la sesión. Después los leeremos desde la carpeta, como haremos con las fuentes del proyecto. Hemos dejado algunos espacios y líneas sobrantes para observar la limpieza.

In [ ]:
carpeta = Path("corpus_clase07")
carpeta.mkdir(exist_ok=True)
# Código de cada documento en el repositorio ficticio del proyecto (Clase 4).
CODIGOS = {"Politica_Permisos.md": "DOC-1002", "Politica_Recursos_Humanos.md": "DOC-1001"}
fuentes = {
    "Politica_Recursos_Humanos.md": """# Política de Recursos Humanos

## Vacaciones

Los empleados deberán solicitar sus vacaciones siguiendo el procedimiento establecido por la organización. La solicitud se registra en el portal de  talento humano con al menos quince días hábiles de anticipación.

Solo pueden tomar vacaciones quienes hayan cumplido un año de servicio. Si la solicitud llega con menos anticipación, no se aprueba automáticamente y queda sujeta a la disponibilidad del área.


El jefe inmediato responde la solicitud en un plazo de cinco días hábiles. Las vacaciones aprobadas no se pueden acumular por más de dos periodos.

## Incapacidades

La incapacidad médica se reporta al jefe inmediato el mismo día. El soporte se entrega a talento humano en los tres días hábiles siguientes; sin él, los días no se registran como incapacidad.

## Vigencia del documento

Este documento se identifica con el código DOC-1001. Su vigencia se consulta en el repositorio documental con ese código; este texto no contiene el estado actual del documento.
""",
    "Politica_Permisos.md": """# Política de Permisos

## Permisos laborales

La solicitud de permisos debe cumplir con los requisitos establecidos en la política correspondiente. El empleado diligencia el formato en el portal de talento humano con dos días hábiles de anticipación e indica el motivo.

Los permisos por calamidad doméstica pueden pedirse el mismo día. Si el soporte no se entrega en los cinco días hábiles siguientes, el permiso se registra como ausencia no justificada.

## Trabajo remoto

El trabajo remoto ocasional requiere aprobación escrita del jefe inmediato antes de la fecha.  Este capítulo está en actualización; mientras tanto no se autorizan más de dos días remotos por semana.

## Vigencia del documento

Este documento se identifica con el código DOC-1002. Su vigencia se consulta en el repositorio documental con ese código; este texto no contiene el estado actual del documento.
""",
}
for nombre, contenido in fuentes.items():
    (carpeta / nombre).write_text(contenido, encoding="utf-8")
print("Archivos:", ", ".join(fuentes))


## 3. Lectura y limpieza
Conservamos archivo, título, versión y huella del original. Reducimos espacios y líneas repetidas; mantenemos párrafos, negaciones y condiciones. Lean la muestra y compárenla con el documento anterior.

In [ ]:
documentos = []
for ruta in sorted(carpeta.glob("*.md")):
    original = ruta.read_text(encoding="utf-8")
    limpio = re.sub(r"[ \t]+", " ", original)
    limpio = re.sub(r"\n{3,}", "\n\n", limpio).strip()
    documentos.append({
        "document_id": ruta.stem, "source": ruta.name,
        "title": limpio.splitlines()[0].lstrip("# "), "version": "1.0-ficticio",
        "codigo": CODIGOS[ruta.name],
        "source_hash": hashlib.sha256(original.encode("utf-8")).hexdigest(),
        "texto": limpio,
    })
    print(f"{ruta.name} ({CODIGOS[ruta.name]}): {len(original)} → {len(limpio)} caracteres")
print("\nMuestra del texto limpio:\n", documentos[0]["texto"][:450])

## 4. Dos estrategias de fragmentación
**Ventanas:** recorremos cada sección por tokens, con solapamiento. **Párrafos:** conservamos sus límites y subdividimos solo los párrafos que exceden el presupuesto. En la segunda estrategia no repetimos texto entre párrafos; sí entre ventanas de un párrafo largo.

Ambas incluyen título y sección en el texto vectorizado. Conservamos el mismo modelo, presupuesto y medida de búsqueda. Revisen los tamaños y el final de los fragmentos: ¿alguna condición quedó separada de su consecuencia?

In [ ]:
def ventanas(texto, limite):
    offsets = tokenizador(texto, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    if limite <= SOLAPAMIENTO:
        raise ValueError("El encabezado deja muy poco espacio para el contenido.")
    for inicio in range(0, len(offsets), limite - SOLAPAMIENTO):
        fin = min(inicio + limite, len(offsets))
        yield texto[offsets[inicio][0]:offsets[fin - 1][1]].strip()
        if fin == len(offsets):
            break

def fragmentar(estrategia):
    registros = []
    for documento in documentos:
        for numero, bloque in enumerate(re.split(r"(?m)^## ", documento["texto"])[1:], 1):
            seccion, cuerpo = bloque.split("\n", 1)
            encabezado = documento["title"] + " / " + seccion + "\n"
            limite = MAX_TOKENS - contar_tokens(encabezado) - 2
            partes = [cuerpo] if estrategia == "ventanas" else re.split(r"\n\s*\n", cuerpo.strip())
            for parte in partes:
                for texto in ventanas(parte.strip(), limite):
                    contenido = encabezado + texto
                    tokens = contar_tokens(contenido)
                    if tokens > min(MAX_TOKENS, modelo.max_seq_length):
                        raise ValueError("Un fragmento supera el límite: ajuste el presupuesto.")
                    registros.append({
                        **{k: documento[k] for k in ["document_id", "codigo", "source", "title", "version", "source_hash"]},
                        "chunk_id": f"{documento['document_id']}:{documento['version']}:{estrategia}:{len(registros)}",
                        "section": seccion, "section_number": numero, "tokens": tokens, "texto": contenido,
                    })
    return registros

fragmentos = {nombre: fragmentar(nombre) for nombre in ["ventanas", "parrafos"]}
for nombre, registros in fragmentos.items():
    print(f"\n{nombre}: {len(registros)} fragmentos; máximo {max(r['tokens'] for r in registros)} tokens")
    for registro in registros[:2]:
        print(registro["chunk_id"], "|", registro["source"], "|", registro["tokens"], "tokens")
        print(registro["texto"], "\n")

## 5. Consulta del usuario y comparación
Calculamos los embeddings de ambos conjuntos una vez. Después escribimos la pregunta en **CONSULTA** y comparamos los tres primeros resultados. Cada resultado conserva el archivo y la sección; vuelvan al original para comprobar su contenido. La similitud coseno ordena candidatos, no mide certeza.

In [ ]:
matrices = {
    nombre: modelo.encode([r["texto"] for r in registros], normalize_embeddings=True, show_progress_bar=False)
    for nombre, registros in fragmentos.items()
}

def buscar(consulta, registros, matriz, k=3):
    if not consulta.strip():
        raise ValueError("Escriba una consulta.")
    if contar_tokens(consulta) > modelo.max_seq_length:
        raise ValueError("La consulta supera el límite del modelo.")
    vector = modelo.encode(consulta, normalize_embeddings=True)
    puntajes = matriz @ vector
    return [(registros[int(i)], float(puntajes[i])) for i in np.argsort(-puntajes)[:k]]

In [ ]:
#@title Escriba aquí la consulta
CONSULTA = "¿Me aprueban las vacaciones si las pido con poca anticipación?" #@param {type:"string"}

for nombre in fragmentos:
    print("\nESTRATEGIA:", nombre)
    for registro, puntaje in buscar(CONSULTA, fragmentos[nombre], matrices[nombre]):
        print(f"{registro['source']} ({registro['codigo']}) · {registro['section']} · coseno {puntaje:.3f}")
        print(registro["chunk_id"])
        print(registro["texto"], "\n")

Prueben también «Tuve una calamidad, ¿puedo pedir el permiso hoy mismo?», «¿Puedo trabajar desde casa toda la semana?» y «¿Cómo solicito un préstamo de nómina?». Antes de ejecutar, señalen dónde debería estar la evidencia. Para préstamos no hay respuesta en este corpus, aunque la búsqueda devuelva candidatos. Anoten qué estrategia conserva mejor la información necesaria en cada caso.

## 6. Guardar y volver a cargar el índice
Elijan una estrategia después de comparar. Guardamos vectores, registros y configuración; luego los leemos nuevamente y repetimos la consulta. Esta colección pequeña utiliza una matriz para búsqueda exacta. En Colab descarguen el ZIP para conservarlo fuera de la sesión.

In [ ]:
ESTRATEGIA = "parrafos" #@param ["ventanas", "parrafos"]
destino = Path("indice_clase07")
destino.mkdir(exist_ok=True)
configuracion = {
    "modelo": MODELO, "revision": REVISION,
    "estrategia": ESTRATEGIA, "max_tokens": MAX_TOKENS, "solapamiento": SOLAPAMIENTO,
    "normalizados": True, "similitud": "coseno", "max_seq_length": modelo.max_seq_length,
    "corpus": {"Politica_Recursos_Humanos.md": "DOC-1001", "Politica_Permisos.md": "DOC-1002"},
    "excluidos": {"Manual_Induccion": "DOC-1003 desactualizado"},
}
np.save(destino / "vectores.npy", matrices[ESTRATEGIA])
(destino / "registros.json").write_text(json.dumps(fragmentos[ESTRATEGIA], ensure_ascii=False, indent=2), encoding="utf-8")
(destino / "configuracion.json").write_text(json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8")

# Leemos desde disco: no recalculamos los embeddings del corpus.
registros_cargados = json.loads((destino / "registros.json").read_text(encoding="utf-8"))
vectores_cargados = np.load(destino / "vectores.npy", allow_pickle=False)
config_cargada = json.loads((destino / "configuracion.json").read_text(encoding="utf-8"))
assert config_cargada["modelo"] == MODELO and vectores_cargados.shape[0] == len(registros_cargados)
antes = buscar(CONSULTA, fragmentos[ESTRATEGIA], matrices[ESTRATEGIA])
despues = buscar(CONSULTA, registros_cargados, vectores_cargados)
assert [r["chunk_id"] for r, _ in antes] == [r["chunk_id"] for r, _ in despues]
assert np.allclose([p for _, p in antes], [p for _, p in despues])
print("Índice cargado: mismos fragmentos y puntajes para la consulta.")

with zipfile.ZipFile("Clase_07_Indice_y_Corpus.zip", "w", zipfile.ZIP_DEFLATED) as archivo:
    for ruta in [*destino.iterdir(), *carpeta.glob("*.md")]:
        archivo.write(ruta, arcname=ruta.as_posix())
print("Archivo para conservar:", "Clase_07_Indice_y_Corpus.zip")

Para usar este índice en otra sesión: descompriman el ZIP, lean la configuración y carguen `SentenceTransformer(config_cargada["modelo"], revision=config_cargada["revision"])`. Lean los registros y la matriz como en la celda anterior. Las consultas deben usar esa misma representación.

In [ ]:
#@title Descargar el índice y los documentos (opcional)
DESCARGAR = False #@param {type:"boolean"}
if DESCARGAR:
    from google.colab import files
    files.download("Clase_07_Indice_y_Corpus.zip")

## 7. Trabajo del equipo
Los dos documentos de la demo se reemplazaron por las políticas del proyecto (punto 2), con título, versión, código y origen registrados en cada fragmento. Ahora comparamos las dos estrategias con preguntas escritas **antes** de ver los resultados.

Cada pregunta indica la evidencia esperada (documento y sección) y las frases que deben aparecer en el fragmento para considerarlo suficiente. Las cinco cubren lo que pide la clase:

| # | Tipo | Pregunta | Evidencia esperada |
|---|---|---|---|
| 1 | Paráfrasis | ¿Con cuánto tiempo debo pedir mis vacaciones? | DOC-1001 · Vacaciones |
| 2 | Regla con condición cerca de un corte | ¿Me aprueban las vacaciones si las pido con menos de quince días? | DOC-1001 · Vacaciones (condición y consecuencia juntas) |
| 3 | Excepción a una regla | Tuve una calamidad, ¿puedo pedir el permiso hoy mismo? | DOC-1002 · Permisos laborales |
| 4 | Dato que el corpus no tiene | ¿Está vigente el documento DOC-1002? | Ninguno suficiente: la vigencia sale de la herramienta de la Clase 4 |
| 5 | Fuera de cobertura | ¿Cómo solicito un préstamo de nómina? | Ninguno suficiente |


In [ ]:
pruebas = [
    ("¿Con cuánto tiempo debo pedir mis vacaciones?", "DOC-1001", "Vacaciones", ["quince días hábiles"]),
    ("¿Me aprueban las vacaciones si las pido con menos de quince días?", "DOC-1001", "Vacaciones", ["menos anticipación", "no se aprueba automáticamente"]),
    ("Tuve una calamidad, ¿puedo pedir el permiso hoy mismo?", "DOC-1002", "Permisos laborales", ["calamidad doméstica", "mismo día"]),
    ("¿Está vigente el documento DOC-1002?", None, None, []),
    ("¿Cómo solicito un préstamo de nómina?", None, None, []),
]

for pregunta, codigo, seccion, frases in pruebas:
    esperado = f"{codigo} · {seccion}" if codigo else "ninguno suficiente"
    print(f"\nPregunta: {pregunta} | Esperado: {esperado}")
    for nombre in fragmentos:
        resultados = buscar(pregunta, fragmentos[nombre], matrices[nombre])
        if codigo:
            puestos = [i for i, (r, _) in enumerate(resultados, 1)
                       if r["codigo"] == codigo and r["section"] == seccion and all(f in r["texto"] for f in frases)]
            veredicto = f"evidencia completa en el puesto {puestos[0]}" if puestos else "FALLO: ningún fragmento del top-3 tiene la evidencia completa"
        else:
            veredicto = "revisar: el corpus no contiene la respuesta"
        print(f"  [{nombre}] {veredicto}")
        for registro, puntaje in resultados:
            print(f"     {puntaje:.3f} | {registro['codigo']} · {registro['section']} | {registro['texto'].splitlines()[-1][:110]}")


**Registro para la bitácora** (completar con la ejecución):

| Campo | Registro |
|---|---|
| Corpus | `Politica_Recursos_Humanos.md` (DOC-1001) y `Politica_Permisos.md` (DOC-1002), versión `1.0-ficticio` |
| Excluido | `Manual_Induccion` (DOC-1003), por estar desactualizado |
| Configuración | MiniLM multilingüe, 80 tokens, solapamiento 12, coseno, k = 3 |
| Fragmentos | ventanas: __ · párrafos: __ |
| Resultado útil | |
| Fallo | |
| Estrategia elegida y por qué | |
| Índice guardado | `Clase_07_Indice_y_Corpus.zip` |

**Antes de terminar:** ¿cada fragmento permite volver a la fuente?, ¿se conservan las condiciones del procedimiento?, ¿el índice cargado devuelve los mismos resultados?

Documentación: [modelo de embeddings](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) · [Sentence Transformers](https://sbert.net/docs/package_reference/sentence_transformer/model.html).